In [2]:
from langgraph.graph import StateGraph,START,END
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langgraph.graph import MessagesState
from typing import Annotated
import json

In [74]:
class State(TypedDict):
    """The state that will flow through our Graph"""
    messages: str

Message State

In [ ]:
class State(TypedDict):
    messages:Annotated[list,add_messages]

In [75]:


def node_a(state:State)->dict:
    """Appends a to the state"""
    print("Node A executing")    
    return {"messages":state["messages"] +"a"}

def node_b(state:State)->dict:
    """Appends b to the state"""
    print("Node B executing")
    return {"messages":state["messages"] +"b"}

In [85]:
graph=StateGraph(State)
graph.add_node("node_a",node_a)
graph.add_node("node_b",node_b)

graph.add_edge(START,"node_a")
graph.add_edge("node_a","node_b")
graph.add_edge("node_b",END)

print("Graph Structure Defined")

Graph Structure Defined


In [86]:
import base64
from IPython.display import Image, display

def show_graphical_image(app):
    mermaid_text = app.get_graph().draw_mermaid()
    # Encode the mermaid syntax as base64 and use mermaid.ink as an image source
    graph_bytes  = mermaid_text.encode("utf-8")
    base64_str   = base64.urlsafe_b64encode(graph_bytes).decode("utf-8")
    url          = f"https://mermaid.ink/img/{base64_str}?theme=default"
    return url


In [ ]:

from IPython.display import Image,display

app = graph.compile()
url=show_graphical_image(app)
display(Image(url=url))

In [78]:
initial_state={"messages":""}
result=app.invoke(initial_state)
print(f"\n {'*'*50}")
print(json.dumps(result,indent=2))
print(f"\n {'*'*50}")

Node A executing
Node B executing

 **************************************************
{
  "messages": "ab"
}

 **************************************************


In [89]:
import re

class WordState(TypedDict):
    original:str
    processed:str
    word_count:int

def to_lowercase(state:WordState)->dict:
    """To convert the words to Lower Case"""
    print("Converting to lowecase letters")
    lowered_case=state['original'].lower()
    return {"processed":lowered_case}

def remove_punctuations(state:WordState)->dict:
    """ To remove Punctuations"""
    print("Removing Punctuation Marks")
    new_words=re.sub(r'[^\w\s]','',state['processed'])
    return {"processed":new_words}

def word_count(state:WordState)->dict:
    """To give a word count"""
    print("Calculate Word Count")
    word_list=state['processed'].split()
    return {"word_count":len(word_list)}





In [88]:
graph=StateGraph(WordState)
graph.add_node(to_lowercase,"to_lowercase")
graph.add_node(remove_punctuations,"remove_punctuations")
graph.add_node(word_count,"word_count")

graph.add_edge(START,"to_lowercase")
graph.add_edge("to_lowercase","remove_punctuations")
graph.add_edge("remove_punctuations","word_count")
graph.add_edge("word_count",END)




In [90]:
from IPython.display import Image
app=graph.compile()
url=show_graphical_image(app)
display(Image(url=url))

In [91]:
test_text = "Hello, World! This is LangGraph. It's amazing!!"
initial_state={"original":test_text,"processed":"","word_count":""}
result=app.invoke(initial_state)
print(f"\n {'*'*50}")
print(json.dumps(result,indent=2))
print(f"\n {'*'*50}")

Converting to lowecase letters
Removing Punctuation Marks
Calculate Word Count

 **************************************************
{
  "original": "Hello, World! This is LangGraph. It's amazing!!",
  "processed": "hello world this is langgraph its amazing",
  "word_count": 7
}

 **************************************************
